# Ridge regression with gradient descent

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
warnings.simplefilter("ignore")

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

## scikit-learn: SGDRegressor with an L2 penalty

In [ ]:
sgd = SGDRegressor(penalty="l2", alpha=0.001, max_iter=500, eta0=0.1,
                   learning_rate="constant", random_state=0)
sgd.fit(X_train, y_train)
print("SGDRegressor test R2", round(sgd.score(X_test, y_test), 4))

## scikit-learn: Ridge with an iterative solver

In [ ]:
ridge = Ridge(alpha=0.001, max_iter=500, solver="sparse_cg").fit(X_train, y_train)
print("Ridge sparse_cg test R2", round(ridge.score(X_test, y_test), 4))

## From scratch: batch gradient descent

In [ ]:
class MyRidgeGD:
    def __init__(self, epochs, learning_rate, alpha):
        self.epochs, self.lr, self.alpha = epochs, learning_rate, alpha

    def fit(self, X, y):
        X = np.insert(X, 0, 1, axis=1)
        w = np.insert(np.ones(X.shape[1] - 1), 0, 0.0)   # intercept 0, slopes 1
        I = np.identity(X.shape[1])
        I[0, 0] = 0                                      # intercept not penalised
        for _ in range(self.epochs):
            grad = X.T @ X @ w - X.T @ y + self.alpha * I @ w
            w = w - self.lr * grad
        self.intercept_, self.coef_ = w[0], w[1:]
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

mine = MyRidgeGD(epochs=500, learning_rate=0.005, alpha=0.001).fit(X_train, y_train)
print("MyRidgeGD test R2", round(r2_score(y_test, mine.predict(X_test)), 4))
print("intercept", round(mine.intercept_, 2))

## Too large a learning rate

In [ ]:
big = MyRidgeGD(epochs=100, learning_rate=0.006, alpha=0.001).fit(X_train, y_train)
print("largest coefficient after 100 epochs:", f"{np.abs(big.coef_).max():.3g}")